In [1]:
import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoTokenizer

`pytorch` contains an `LSTM` network pre-baked. before using this pre-baked `LSTM` class, i consider developing my own module. an RNN improves on a FNN for problems which require knowledge of previous states, and/or include a time domain. they do this by sharing weights within a network across each 'iteration' (a day, a word, whatever). an LSTM has three main components:
- $c_{t-1}$: the cell state at recurrence $t-1$
- $h_{t-1}$: the hidden output at recurrence $t-1$
- $x_t$: the current input

thus, the output at recurrence $t$, $h_t$ is defined by $h_t = g(x_t, h_{t-1}, c_{t-1})$. within each cell, there exists a mechanism to *forget* previous cell state information and then *update* new cell state information into $c$. the cell state $c$ is shared within a layer across recurrences and is continually updated, thus for $N$ LSTM cells, there exists $N$ cell state variables.

In [2]:
class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        # construct the parent class
        super().__init__()
        
        # for the affine linear transformation of previous h and current x
        # 4 components x hidden_size values each
        # since y = XW + b where X = [batch_size, input_size] and W = [input_size * hidden_size]
        # thus when called, can handle a batch of N samples (with N * hidden_size output)
        self.x_trans = nn.Linear(input_size, 4 * hidden_size)
        self.h_trans = nn.Linear(hidden_size, 4 * hidden_size)

    def forward(self, x_curr: torch.Tensor, h_prev: torch.Tensor, c_prev: torch.Tensor):
        pre_gates: torch.Tensor = self.x_trans(x_curr) + self.h_trans(h_prev)

        # forget, input, update and output
        # forget: multiplies with the previous cell state; how much to forget
        # input: multiplier for the update
        # update: the actual value to update into the new cell state
        # output: the multipler for how much of the previous hidden state to carry forward
        fgt, inp, upd, out = pre_gates.chunk(4, dim=-1)

        fgt, inp, upd, out = torch.sigmoid(fgt), torch.sigmoid(inp), torch.tanh(upd), torch.sigmoid(out)

        c_curr = fgt * c_prev + inp * upd
        h_curr = out * torch.tanh(c_curr)

        return h_curr, c_curr

the above is good for just one recurrence at one cell, so we'll need a wrapper class to be able to accept an input $X \in \mathbb{R}^{M \times T}$ for $M$ features and $T$ recurrences. Since we wish to train on more than one sample, our input tensor will indeeed be three-dimensional $X_{train} \in \mathbb{R}^{N \times M \times T}$ for $N$ samples.

In [3]:
class LSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()

        self.hidden_size = hidden_size
        self.cell = LSTMCell(input_size, hidden_size)

    def forward(self, x: torch.Tensor):
        """
        x is a 3d tensor [batch_size, sequence_length, input_size]
        """
        batch_size = x.shape[0]
        sequence_length = x.shape[1]

        # each sequence (sample) requires a hidden state and cell state of size hidden size
        h = torch.zeros(batch_size, self.hidden_size, device = x.device)
        c = torch.zeros(batch_size, self.hidden_size, device = x.device)

        outputs = []

        # we train the network at each sequence t.
        # an obvious question is - what about sequences of different lengths?
        # the below is essentially a cross-section at some time t for all of the samples
        for t in range(sequence_length):
            x_t = x[:, t, :]

            # in this case, h is of dimension [batch_size * hidden_size]. thus the LSTMCell
            # will output [batch_size * (4 * hidden_size)] for the components
            h, c = self.cell(x_t, h, c)

            outputs.append(h)

        # we iterated on dimension 1 (the sequence length), thus we should stack the outputs by dimension 1
        outputs = torch.stack(outputs, dim=1)

        return outputs, (h, c)

we test the implementation with a very simple exercise - remember the first value. for example:
```
[1, 0, 0, 1, 1, 1, 1, 0, 0] -> 1
[0, 0, 0, 1, 1, 1, 1, 1, 1] -> 0
```
here our input size is `1` (a numeric value). only allowing `1`s and `0`s makes this a binary classification problem.

In [27]:
import numpy as np
from sklearn.model_selection import train_test_split

def p_remember_first_value(n_samples: int, n_sequence: int, tr_size=0.8):
    """
    returns (train data, test data) tuple of dimension (n_samples, n_sequence + 1) where the last column is the target column (class).
    """
    data = np.random.randint(2, size=(n_samples, n_sequence))
    tgt_col = data[:, 0]
    data = np.hstack((data, tgt_col.reshape(-1, 1)))

    tr_data, tst_data = train_test_split(data, train_size=tr_size)

    return tr_data, tst_data

our final step is to declare the binary classifier using our LSTM. we start with a one layer LSTM network.

In [35]:
from torch.utils.data import TensorDataset, DataLoader

class BinaryLSTMClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()

        self.lstm = LSTM(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor):
        outputs, (h, c) = self.lstm(x)
        logits = self.output(h)
        return logits

# define network
model = BinaryLSTMClassifier(1, 32)
tr_data, tst_data = p_remember_first_value(10000, 32)
X_tr, y_tr = torch.tensor(tr_data[:, :-1]), torch.tensor(tr_data[:, -1])
X_tst, y_Tst = torch.tensor(tst_data[:, :-1]), torch.tensor(tst_data[:, -1])

# train
train_dataset = TensorDataset(X_tr, y_tr)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

loss_fn = torch.nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters())